# 04 Outbreak Investigation Workflow — Reference Solutions

Complete solutions to the Pine and Cypress Nursing Home Legionnaires' disease cluster SitRep exercises.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (avoid CJK labels rendering as boxes) --
# Scan the system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150


## Question 1: Summary metrics

In [ ]:
df = pd.read_csv("data/synthetic/legionella_outbreak.csv")

# Convert dates
date_cols = [
    "facility_admission_date", "symptom_onset_date",
    "hospitalization_date", "death_date", "notification_date",
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

total = len(df)
infected = df["infected"].sum()
confirmed = (df["case_classification"] == "confirmed").sum()
probable = (df["case_classification"] == "probable").sum()
hospitalized = df["hospitalized"].sum()
icu = df["icu_admission"].sum()
deaths = (df["outcome"] == "dead").sum()

print("=" * 50)
print("Pine and Cypress Nursing Home Legionnaires' Disease Cluster — SitRep")
print("=" * 50)
print(f"Total residents: {total}")
print(f"Infected: {infected} (attack rate {infected/total:.1%})")
print(f"  Confirmed: {confirmed}   Probable: {probable}")
print(f"Hospitalized: {hospitalized} (hospitalization rate {hospitalized/infected:.1%})")
print(f"ICU: {icu} (ICU rate {icu/hospitalized:.1%})")
print(f"Deaths: {deaths} (CFR {deaths/infected:.1%})")

## Question 2: The person/time/place trio

In [ ]:
# --- Person ---
cases = df[df["infected"] == 1]

print("=== Demographic characteristics (infected) ===")
print(f"Median age: {cases['age'].median():.0f} years"
      f" (range {cases['age'].min()}-{cases['age'].max()})")
print(f"Proportion male: {(cases['sex'] == 'M').mean():.1%}")

In [ ]:
import matplotlib.dates as mdates

# --- Time ---
daily = cases.groupby("symptom_onset_date").size().rename("cases")

# Add the pre-outbreak baseline period (including zero-case days)
date_range = pd.date_range(
    daily.index.min() - pd.Timedelta(days=3),
    daily.index.max() + pd.Timedelta(days=1),
)
daily = daily.reindex(date_range, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    daily.index, daily.values,
    width=1.0,
    color="#2c7fb8", edgecolor="white", linewidth=0.5,
)
ax.set_title(
    "Pine and Cypress Nursing Home Legionnaires' Disease Epidemic Curve, by Onset Date, January 2026",
    fontsize=13, fontweight="bold",
)
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d"))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
fig.autofmt_xdate(rotation=45, ha="right")

ax.set_ylim(bottom=0)
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

# Use the original daily series (without the baseline period) to find the peak
daily_cases = cases.groupby("symptom_onset_date").size().rename("cases")
print(f"Outbreak period: {daily_cases.index.min().date()} – {daily_cases.index.max().date()}")
print(f"Peak day: {daily_cases.idxmax().date()} ({daily_cases.max()} cases)")

In [ ]:
# --- Place ---
wing_stats = (
    df.groupby(["floor", "wing"])
    .agg(
        residents=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
    .reset_index()
)
wing_stats["AR%"] = (wing_stats["infected"] / wing_stats["residents"] * 100).round(1)
wing_stats["CFR%"] = (wing_stats["deaths"] / wing_stats["infected"] * 100).round(1)
wing_stats["label"] = wing_stats["floor"].astype(str) + wing_stats["wing"]

print("=== Outbreak summary by wing ===")
print(wing_stats[["label", "residents", "infected", "AR%", "deaths", "CFR%"]]
      .to_string(index=False))

## Question 3: Stratified summary by age group

In [ ]:
df["age_group"] = pd.cut(
    df["age"], bins=[59, 69, 79, 89, 100],
    labels=["60-69", "70-79", "80-89", "90+"],
)

age_stats = (
    df.groupby("age_group", observed=True)
    .agg(
        residents=("case_id", "size"),
        infected=("infected", "sum"),
        deaths=("outcome", lambda x: (x == "dead").sum()),
    )
)
age_stats["AR%"] = (age_stats["infected"] / age_stats["residents"] * 100).round(1)
age_stats["CFR%"] = (age_stats["deaths"] / age_stats["infected"] * 100).round(1)

print("=== Age group stratified summary ===")
print(age_stats.to_string())

print(f"\nHighest attack rate: {age_stats['AR%'].idxmax()} ({age_stats['AR%'].max()}%)")
print(f"Highest CFR: {age_stats['CFR%'].idxmax()} ({age_stats['CFR%'].max()}%)")
print("→ The age group with the highest attack rate isn't necessarily the one with the highest CFR —")
print("  the attack rate reflects 'risk of infection', while CFR reflects 'prognosis after infection'; the two are driven by different factors.")

## Question 4 (challenge): the generate_sitrep function

In [ ]:
def generate_sitrep(csv_path):
    """Produce a SitRep summary dictionary from a CSV."""
    df = pd.read_csv(csv_path)
    for col in ["symptom_onset_date", "hospitalization_date",
                "death_date", "notification_date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

    total = len(df)
    infected_n = int(df["infected"].sum())
    deaths_n = int((df["outcome"] == "dead").sum())

    # peak day
    cases = df[df["infected"] == 1]
    daily = cases.groupby("symptom_onset_date").size()
    peak_date = str(daily.idxmax().date()) if len(daily) > 0 else None

    # wing with the highest attack rate
    ws = (
        df.groupby(["floor", "wing"])
        .agg(residents=("case_id", "size"), infected=("infected", "sum"))
        .reset_index()
    )
    ws["ar"] = ws["infected"] / ws["residents"]
    worst = ws.loc[ws["ar"].idxmax()]
    worst_wing = f"{worst['floor']}{worst['wing']}"

    return {
        "total_residents": total,
        "infected": infected_n,
        "attack_rate": round(infected_n / total * 100, 1),
        "deaths": deaths_n,
        "cfr": round(deaths_n / infected_n * 100, 1) if infected_n else 0,
        "hospitalized": int(df["hospitalized"].sum()),
        "icu": int(df["icu_admission"].sum()),
        "peak_date": peak_date,
        "worst_wing": worst_wing,
    }

result = generate_sitrep("data/synthetic/legionella_outbreak.csv")
print("=== SitRep structured output ===")
for k, v in result.items():
    print(f"  {k}: {v}")

## Question 5: Produce a Word report

In [ ]:
from io import BytesIO
from datetime import datetime
from docx import Document
from docx.shared import Inches

pathlib.Path("output").mkdir(exist_ok=True)

# Save the epidemic curve into memory
epicurve_buf = BytesIO()
fig.savefig(epicurve_buf, format="png", dpi=150, bbox_inches="tight")
epicurve_buf.seek(0)

report_time = datetime.now().strftime("%Y-%m-%d %H:%M")

doc = Document()
doc.add_heading("Pine and Cypress Nursing Home Legionnaires' SitRep", level=1)
doc.add_paragraph(f"Report time: {report_time}")

# Summary metrics table
doc.add_heading("Summary metrics", level=2)
table = doc.add_table(rows=5, cols=2, style="Light Grid Accent 1")
for i, (label, value) in enumerate([
    ("Total residents", str(total)),
    ("Infected", f"{infected} (attack rate {infected/total:.1%})"),
    ("Confirmed / Probable", f"{confirmed} / {probable}"),
    ("Hospitalized / ICU", f"{hospitalized} / {icu}"),
    ("Deaths", f"{deaths} (CFR {deaths/infected:.1%})"),
]):
    table.rows[i].cells[0].text = label
    table.rows[i].cells[1].text = value

# Embed the epidemic curve
doc.add_heading("Epidemic curve", level=2)
epicurve_buf.seek(0)
doc.add_picture(epicurve_buf, width=Inches(6))

doc.save("output/my_sitrep.docx")
print("Word report saved: output/my_sitrep.docx")

### Interpretation

- **Attack rate ~43%**: very high, indicating a severe outbreak with a widespread exposure source
- **CFR ~16%**: Legionnaires' disease has an elevated CFR in nursing-home populations, consistent with the literature
- **Wing 3B has the highest attack rate**: prioritize investigating that wing's water supply and shower equipment
- **Age group differences**: the highest attack rate and the highest CFR may not fall in the same age group, showing that "risk of infection" and "prognosis" are driven by different factors

## Question 6 Solution

In [ ]:
import numpy as np
from epi_learning.metrics import attack_rate, risk_ratio
from epi_learning.viz import plot_epi_curve

# Data: guest list from a community banquet (simulated norovirus foodborne cluster)
rng = np.random.default_rng(614)
n = 240
banquet_date = pd.Timestamp("2026-03-14")

guest_id = np.arange(1, n + 1)
table_no = rng.integers(1, 25, n)  # 24 tables
ate_oysters = rng.random(n) < 0.4  # 40% of guests ate the cold oyster platter

# Guests who ate the cold oyster platter have a markedly higher infection probability (suspected exposure)
p_infect = np.where(ate_oysters, 0.65, 0.08)
infected = rng.random(n) < p_infect

# Norovirus has a short incubation period (about 12-48 hours); only cases have an onset date
incubation_hours = rng.normal(30, 8, n).clip(10, 60)
onset_datetime = pd.DatetimeIndex(banquet_date + pd.to_timedelta(incubation_hours, unit="h"))

df6 = pd.DataFrame({
    "guest_id": guest_id,
    "table_no": table_no,
    "ate_oysters": np.where(ate_oysters, "yes", "no"),
    "infected": infected,
    "symptom_onset_date": pd.NaT,
})
df6.loc[infected, "symptom_onset_date"] = onset_datetime[infected].normalize()

# --- Exposure table: whether the guest ate the oyster platter ---
exposed = df6[df6["ate_oysters"] == "yes"]
unexposed = df6[df6["ate_oysters"] == "no"]
exposed_cases, exposed_total = int(exposed["infected"].sum()), len(exposed)
unexposed_cases, unexposed_total = int(unexposed["infected"].sum()), len(unexposed)

ar_exposed = attack_rate(exposed_cases, exposed_total)
ar_unexposed = attack_rate(unexposed_cases, unexposed_total)
rr = risk_ratio(exposed_cases, exposed_total, unexposed_cases, unexposed_total)

print("=== Exposure table: ate oyster platter ===")
print(f"Ate oysters:  {exposed_cases}/{exposed_total}   attack rate {ar_exposed:.1%}")
print(f"Did not eat:  {unexposed_cases}/{unexposed_total}   attack rate {ar_unexposed:.1%}")
print(f"Risk ratio RR = {rr:.2f}")

# --- Epidemic curve ---
ax = plot_epi_curve(df6.dropna(subset=["symptom_onset_date"]), date_col="symptom_onset_date")
ax.set_title("Banquet Norovirus Cluster Epidemic Curve, by Onset Date")
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")
plt.show()

# --- SitRep summary ---
total_cases = int(df6["infected"].sum())
print("\n=== SitRep Summary ===")
print(f"Total guests: {n}")
print(f"Cases: {total_cases} (overall attack rate {attack_rate(total_cases, n):.1%})")
print(f"Risk ratio RR (ate oyster platter vs did not) = {rr:.2f}")

print("\nInterpretation: the epidemic curve shows a single peak within 1-2 days after the banquet, followed by a rapid decline,")
print("consistent with a typical 'point source' cluster pattern;")
print(f"guests who ate the cold oyster platter had {rr:.1f} times the infection risk of those who did not, supporting the oyster platter as the suspected exposure source of this cluster.")

## Question 7 Solution

In [ ]:
import numpy as np
from epi_learning.metrics import risk_ratio
from epi_learning.tabulate import summarize_by_group
from epi_learning.viz import plot_epi_curve

# Data: employee roster after a company department dinner (simulated COVID-19 workplace cluster)
rng = np.random.default_rng(719)
n = 450
departments = ["業務部", "行政部", "IT部", "財務部", "客服部"]
dept_sizes = [150, 80, 70, 60, 90]
department = np.repeat(departments, dept_sizes)

# Attendance rate differs by department (業務部 has the highest attendance rate)
p_meeting = {"業務部": 0.75, "行政部": 0.30, "IT部": 0.15, "財務部": 0.20, "客服部": 0.35}
attended_meeting = np.array([rng.random() < p_meeting[d] for d in department])

# Attendees have a markedly higher infection probability
p_infect = np.where(attended_meeting, 0.55, 0.05)
infected = rng.random(n) < p_infect

# COVID-19 incubation period is about 2-8 days
onset_offset_days = rng.integers(2, 9, n)
meeting_date = pd.Timestamp("2026-03-01")
onset_date = meeting_date + pd.to_timedelta(onset_offset_days, unit="D")

df7 = pd.DataFrame({
    "employee_id": np.arange(1, n + 1),
    "department": department,
    "attended_meeting": np.where(attended_meeting, "yes", "no"),
    "infected": infected,
    "symptom_onset_date": pd.NaT,
})
df7.loc[infected, "symptom_onset_date"] = onset_date[infected]

# --- Case counts and share by department ---
cases7 = df7[df7["infected"]]
dept_summary = summarize_by_group(cases7, "department", "employee_id")
print("=== Case counts and share by department ===")
print(dept_summary.to_string(index=False))

# --- Attack rate by department ---
dept_stats = df7.groupby("department").agg(
    n=("employee_id", "size"), infected=("infected", "sum"),
)
dept_stats["AR%"] = (dept_stats["infected"] / dept_stats["n"] * 100).round(1)
dept_stats = dept_stats.sort_values("AR%", ascending=False)
print("\n=== Attack rate by department (highest to lowest) ===")
print(dept_stats.to_string())

# --- Epidemic curve ---
ax = plot_epi_curve(df7.dropna(subset=["symptom_onset_date"]), date_col="symptom_onset_date")
ax.set_title("Company COVID-19 Cluster Epidemic Curve, by Onset Date")
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")
plt.show()

# --- Risk ratio: meeting attendance ---
exp7 = df7[df7["attended_meeting"] == "yes"]
unexp7 = df7[df7["attended_meeting"] == "no"]
rr7 = risk_ratio(
    int(exp7["infected"].sum()), len(exp7),
    int(unexp7["infected"].sum()), len(unexp7),
)

worst_dept = dept_stats.index[0]
print(f"\nDepartment with highest attack rate: {worst_dept} ({dept_stats.iloc[0]['AR%']}%)")
print(f"Risk ratio RR (attended dinner vs did not) = {rr7:.2f}")
print("\nInterpretation: the department with the highest attack rate is also the department with the highest dinner attendance rate,")
print("and attendees have a significantly higher infection risk than non-attendees,")
print("supporting the department dinner as the main transmission hotspot of this workplace cluster; contact tracing should prioritize dinner attendees.")

## Question 8 Solution

In [ ]:
import numpy as np
from epi_learning.metrics import attack_rate, risk_ratio
from epi_learning.viz import plot_epi_curve

# Data: measles cluster at an elementary school, simulating the within-classroom transmission chain (with generation and infector)
rng = np.random.default_rng(2026)
n_classes = 20
students_per_class = 20
n = n_classes * students_per_class  # 400 students

class_id = np.repeat([f"C{i+1:02d}" for i in range(n_classes)], students_per_class)
student_id = np.arange(1, n + 1)
vaccinated = rng.random(n) < 0.92  # 92% vaccine coverage (below the ~95% measles herd immunity threshold)

df8 = pd.DataFrame({
    "student_id": student_id,
    "class_id": class_id,
    "vaccinated": np.where(vaccinated, "yes", "no"),
})
df8["infected"] = False
df8["symptom_onset_date"] = pd.NaT
df8["infector_id"] = pd.NA
df8["generation"] = pd.NA

start_date = pd.Timestamp("2026-03-02")

# Generation 0: 3 unvaccinated community-acquired index cases
unvacc_idx = df8.index[df8["vaccinated"] == "no"].to_numpy()
primary_idx = rng.choice(unvacc_idx, size=3, replace=False)
for idx in primary_idx:
    df8.loc[idx, "infected"] = True
    df8.loc[idx, "symptom_onset_date"] = start_date + pd.Timedelta(days=int(rng.integers(0, 3)))
    df8.loc[idx, "generation"] = 0

# Simulate the within-classroom transmission chain by generation: each case may infect
# uninfected classmates; vaccinated classmates can still be infected, but at a much lower
# probability due to vaccine efficacy (97%)
vaccine_efficacy = 0.97
p_transmit_unvacc = 0.55  # transmission probability per classmate-pair contact (unvaccinated)
max_generations = 5

current_gen = 0
while current_gen < max_generations:
    infectors = df8[(df8["generation"] == current_gen) & df8["infected"]]
    if infectors.empty:
        break
    for _, case in infectors.iterrows():
        classmates = df8[
            (df8["class_id"] == case["class_id"])
            & (~df8["infected"])
            & (df8.index != case.name)
        ]
        for cm_idx, cm in classmates.iterrows():
            transmit_prob = (
                p_transmit_unvacc * (1 - vaccine_efficacy)
                if cm["vaccinated"] == "yes"
                else p_transmit_unvacc
            )
            if rng.random() < transmit_prob:
                generation_interval = max(7, rng.normal(12, 2))  # serial interval in days
                onset = case["symptom_onset_date"] + pd.Timedelta(days=generation_interval)
                df8.loc[cm_idx, "infected"] = True
                df8.loc[cm_idx, "symptom_onset_date"] = onset
                df8.loc[cm_idx, "infector_id"] = case["student_id"]
                df8.loc[cm_idx, "generation"] = current_gen + 1
    current_gen += 1

# --- Attack rate and risk ratio: unvaccinated vs vaccinated ---
unvacc = df8[df8["vaccinated"] == "no"]
vacc = df8[df8["vaccinated"] == "yes"]
unvacc_cases, unvacc_total = int(unvacc["infected"].sum()), len(unvacc)
vacc_cases, vacc_total = int(vacc["infected"].sum()), len(vacc)

ar_unvacc = attack_rate(unvacc_cases, unvacc_total)
ar_vacc = attack_rate(vacc_cases, vacc_total)
rr8 = risk_ratio(unvacc_cases, unvacc_total, vacc_cases, vacc_total)

print("=== Attack rate: vaccination status ===")
print(f"Unvaccinated: {unvacc_cases}/{unvacc_total}   attack rate {ar_unvacc:.1%}")
print(f"Vaccinated:   {vacc_cases}/{vacc_total}   attack rate {ar_vacc:.1%}")
print(f"Risk ratio RR (unvaccinated vs vaccinated) = {rr8:.1f}")

# --- Epidemic curve (observe multiple generational waves) ---
ax = plot_epi_curve(df8.dropna(subset=["symptom_onset_date"]), date_col="symptom_onset_date")
ax.set_title("School Measles Cluster Epidemic Curve, by Onset Date")
ax.set_xlabel("Date of Symptom Onset")
ax.set_ylabel("Number of Cases")
plt.show()

# --- Serial interval estimation ---
secondary = df8.dropna(subset=["infector_id"]).copy()
secondary["infector_id"] = secondary["infector_id"].astype(int)
onset_by_id = df8.set_index("student_id")["symptom_onset_date"]
secondary["infector_onset"] = secondary["infector_id"].map(onset_by_id)
secondary["serial_interval_days"] = (
    secondary["symptom_onset_date"] - secondary["infector_onset"]
).dt.days

si_mean = secondary["serial_interval_days"].mean()
si_median = secondary["serial_interval_days"].median()

# --- SitRep summary ---
total_cases8 = int(df8["infected"].sum())
print("\n=== SitRep Summary ===")
print(f"Total students: {n}")
print(f"Cases: {total_cases8} (overall attack rate {attack_rate(total_cases8, n):.1%})")
print(f"Risk ratio RR (unvaccinated vs vaccinated) = {rr8:.1f}")
print(f"Serial interval estimate: mean {si_mean:.1f} days, median {si_median:.1f} days (n={len(secondary)} transmission pairs)")

print("\nInterpretation:")
print("- Unvaccinated students had a much higher infection risk than vaccinated students, showing vaccination remains the most effective protection")
print("- The estimated serial interval is close to the literature-reported measles serial interval (about 11-12 days),")
print("  indicating that this cluster indeed spread through sustained classroom contact rather than a single point exposure")
print("- The 92% vaccine coverage is below the ~95% threshold needed to establish herd immunity against measles,")
print("  which is a key reason the school cluster was able to keep spreading across generations")